# Partial tracking

A partial tracker picks peaks in every spectrogram frame, then links them
across frames into frequency and amplitude paths. Test tones with known
answers expose its failures before the Saariaho flute excerpt puts it next to
PM2. Read `waveforms_sampling_dft` first for windows and zero padding, and
`interop_partiels_sdif` for the PM2 files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import audiospylt as asp

sr = 44100
BH = "blackmanharris"

def tone(freq, amp=0.5, dur=1.0):
    t = np.arange(round(dur * sr)) / sr
    return amp * np.sin(2 * np.pi * freq * t)

def count(tr):
    return tr.track_id.nunique()

def rms_db(audio):
    if audio.size == 0 or not np.any(audio):
        return -np.inf
    return 20 * np.log10(np.sqrt(np.mean(audio ** 2)))

## Peaks in every frame

Two sines give a known answer. The frame length is 2048 samples; `hop=256`
sets the spacing between frames and `zero_pad=4` samples each spectrum more
finely, without resolving closer tones. These match `track_partials`, rather
than `spectrogram`'s defaults of hop 512 and zero padding 1. `min_db=-80`
rejects quieter peaks in dB re full scale; `spectrogram_peaks` itself defaults
to −60 dB.

In [ ]:
x = tone(440) + tone(1003.1, 0.25)
default_spec = asp.spectrogram(x, sr)
print(f"spectrogram defaults: hop {default_spec.hop}, "
      f"zero_pad {default_spec.n_fft // default_spec.frame_length}")
spec = asp.spectrogram(x, sr, hop=256, zero_pad=4)
peaks = asp.spectrogram_peaks(spec, min_db=-80, prominence_db=20)
print(f"{len(peaks)} rows; kind: {asp.infer_kind(peaks)}")
peaks.head()

371 rows, with frame number, time, frequency, amplitude, FFT bin and
prominence. Peaks do not yet have an identity across frames. The default
spectrogram overlay draws a peaks table as markers: lines would join unrelated
peaks, so asking for `overlay_mode="lines"` raises `ValueError`.

In [ ]:
asp.nb.show(asp.viz.plot_spectrogram(spec, overlay=peaks, fmax=2000))
try:
    asp.viz.plot_spectrogram(spec, overlay=peaks, fmax=2000, overlay_mode="lines")
except ValueError as exc:
    print(f"{type(exc).__name__}: {exc}")

The markers show every surviving peak, including brief ones at the edges.
`link_peaks` makes one weighted assignment per frame: a link gains
`max_cents` minus its pitch distance from the last measured frequency; links
at or beyond the gate are excluded, and peaks can stay unlinked. Its gate is
50 cents by default. `min_duration` keeps only tracks whose last time minus
first time reaches the given span: its own default is 0.05 s, while
`track_partials` uses 0.1 s.

In [ ]:
tracks = asp.link_peaks(peaks, min_duration=0.1)
print(f"{count(tracks)} tracks, {len(tracks)} rows")
print("same chain:", tracks.equals(asp.track_partials(x, sr)))
Path("tsv").mkdir(exist_ok=True)
asp.write_table(peaks, "tsv/two_tones_peaks.tsv")
print(Path("tsv/two_tones_peaks.tsv").read_text().splitlines()[0])

2 tracks, 346 rows, and `True`: `track_partials` is exactly this chain. The
TSV starts with `# audiospylt-table: peaks v1`, so a reader can identify the
kind. `help(asp.track_partials)` lists the rest: `rel_db` is a threshold
relative to each frame's loudest survivor, `fmin` and `fmax` bound the kept
frequencies, and `min_distance_hz` keeps louder peaks when neighbours are too
close. `center=True` places frame centres on the hop grid by padding the ends.

## Side lobes and prominence

A window spreads a sine's spectrum into a main lobe and side lobes.
`prominence_db` measures a peak's height above its surrounding troughs;
`None` removes that floor. Before you run the cell: one sine, how many tracks
will Hann give without a prominence floor?

In [ ]:
x = tone(1000)
rows = []
for label, tr in [
    ("Hann, no prominence floor", asp.track_partials(x, sr, prominence_db=None)),
    ("Hann, 20 dB", asp.track_partials(x, sr)),
    ("Blackman-Harris, no floor", asp.track_partials(x, sr, window=BH, prominence_db=None)),
]:
    rows.append({"setting": label, "tracks": count(tr)})
pd.DataFrame(rows)

Hann without the floor gives 21 tracks from one sine; the default 20 dB
floor leaves 1. Blackman-Harris also gives 1 without a prominence floor. A
side lobe is as steady as the sine, so `min_duration` cannot remove it.

## On the zero-padded grid

The prominence floor has a blind spot. A sine exactly on the FFT grid makes
the sampled spectrum hit exact nulls between side lobes. Those nulls are
clipped at `floor_db` (−120 dB in `spectrogram_peaks`), leaving side lobes far
above the troughs and able to pass the prominence test.

In [ ]:
step = sr / (2048 * 4)
print(f"FFT grid step {step:.2f} Hz; on-grid tone {200 * step:.2f} Hz")
rows = []
for freq, window in [(200 * step, "hann"), (200 * step, BH), (1078.0, "hann")]:
    tr = asp.track_partials(tone(freq), sr, window=window)
    rows.append({"Hz": round(freq, 2), "window": window, "tracks": count(tr)})
pd.DataFrame(rows)

The grid step is 5.38 Hz. At 1076.66 Hz Hann gives 21 tracks even with the
20 dB floor; Blackman-Harris gives 1. At 1078.0 Hz, about a quarter step off
the grid, Hann gives 1. Measure the side lobes of one frame, then raise
`floor_db` to make the troughs shallower.

In [ ]:
for freq in [200 * step, 1078.0]:
    spec = asp.spectrogram(tone(freq), sr, hop=256, zero_pad=4)
    peaks = asp.spectrogram_peaks(spec, min_db=-80)
    frame = peaks[peaks.frame == peaks.frame.max() // 2]
    side = frame[(frame.freq - freq).abs() > 50]
    print(f"{freq:.2f} Hz: {len(side)} side lobes, strongest prominence "
          f"{side.prominence.max():.1f} dB")
    for floor in [-120, -90, -80]:
        kept = asp.spectrogram_peaks(spec, min_db=-80, prominence_db=20, floor_db=floor)
        print(f"  floor_db={floor}: {count(asp.link_peaks(kept, min_duration=0.1))} tracks")

On the grid the strongest side lobe stands 81.8 dB above its troughs; a
quarter step off it stands 12.9 dB, below the 20 dB floor. Raising `floor_db`
makes the troughs shallower: on the grid 21 tracks fall to 13 at −90 dB and 7
at −80 dB, while off the grid the count stays 1. Sweep one grid step to find
how close is too close.

In [ ]:
fs = (200 + np.arange(40) / 40) * step
counts = {}
for window in ["hann", BH]:
    counts[window] = np.array([
        count(asp.track_partials(tone(f, dur=0.5), sr, window=window)) for f in fs
    ])
    print(f"{window}: extra tracks at {(counts[window] > 1).sum()} of {len(fs)} frequencies")
distance = np.minimum(fs - 200 * step, 201 * step - fs)
affected = counts["hann"] > 1
print(f"furthest affected: {distance[affected].max():.2f} Hz from grid")
print(f"nearest unaffected: {distance[~affected].min():.2f} Hz from grid")

Hann has extra tracks at 9 of 40 frequencies; Blackman-Harris has them at
0. The furthest affected sine is 0.54 Hz from the grid, the nearest unaffected
one 0.67 Hz away. Blackman-Harris avoids this failure but has a main lobe
twice as wide as Hann's. Try two equal sines to see the cost.

In [ ]:
rows = []
for d in [100, 120]:
    x = tone(1003.1, 0.3) + tone(1003.1 + d, 0.3)
    rows.append({"separation_Hz": d,
                 "Hann_tracks": count(asp.track_partials(x, sr)),
                 "BH_tracks": count(asp.track_partials(x, sr, window=BH))})
pd.DataFrame(rows)

At a separation of 100 Hz, Hann keeps 2 tracks and Blackman-Harris only 1;
at 120 Hz both keep 2. Use `window="blackmanharris"` for stationary or
synthetic tones. Hann stays the default for recordings because its main lobe
is half as wide, separating closer partials.

## Gaps

`max_gap` counts missed frames before a track closes; it does not count
samples of silence. Zero part of a sine and keep the Blackman-Harris window.
Before you run the cell: which hole, 30 ms or 80 ms, will split the track at
`max_gap=0`?

In [ ]:
rows = []
for ms in [30, 80]:
    x = tone(660)
    hole = round(ms * sr / 1000)
    x[sr // 2:sr // 2 + hole] = 0
    for gap in ([0] if ms == 30 else [0, 4, 7, 8]):
        tr = asp.track_partials(x, sr, window=BH, max_gap=gap)
        rows.append({"hole_ms": ms, "max_gap": gap, "tracks": count(tr)})
pd.DataFrame(rows)

The 30 ms hole leaves 1 track at `max_gap=0`. The 80 ms hole gives 2 tracks
at `max_gap=0`, 4 and 7, but 1 at 8. Count the actual frames without a peak
in the longer hole.

In [ ]:
spec = asp.spectrogram(x, sr, window=BH, hop=256, zero_pad=4)
peaks = asp.spectrogram_peaks(spec, min_db=-80, prominence_db=20)
missing = ~np.isin(np.arange(len(spec.times)), peaks.frame.unique())
edges = np.diff(np.r_[0, missing.astype(int), 0])
runs = np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)
print(f"largest run without a peak: {runs.max()} frames")
print(f"frame: {spec.frame_length} samples, {1000 * spec.frame_length / sr:.1f} ms")

The largest run is 8 frames. A frame lasts 46.4 ms (2048 samples), longer
than the 30 ms hole, so every frame still holds some sine. A bridged track
has no measured rows in the gap: `tracks_to_events` and `synthesize` join its
endpoints with a linear glide in frequency and amplitude. This fills a hole
rather than restoring its missing sound.

## Crossing glides

Two glides exchange frequencies over 2 s. Near their crossing they share a
main lobe, so the peak picker cannot keep them separate. Blackman-Harris
makes this loss visible without the side lobes of Hann.

In [ ]:
t = np.arange(2 * sr) / sr
f1 = 500 * 2 ** (t / 2)
f2 = 1000 * 2 ** (-t / 2)
x = 0.3 * np.sin(2 * np.pi * np.cumsum(f1) / sr)
x += 0.3 * np.sin(2 * np.pi * np.cumsum(f2) / sr)
spec = asp.spectrogram(x, sr, window=BH, hop=256, zero_pad=4)
peaks = asp.spectrogram_peaks(spec, min_db=-80, prominence_db=20)
one = peaks.groupby("frame").filter(lambda g: len(g) == 1)
print(f"{len(peaks)} peaks; {len(one)} frames with one peak, "
      f"{one.time.min():.2f}–{one.time.max():.2f} s")
split = asp.link_peaks(peaks, min_duration=0.1)
print(f"max_gap=0: {count(split)} tracks")
asp.nb.show(asp.viz.plot_events(split))

617 peaks, with only one peak in 73 frames from 0.77 to 1.22 s. The linker
leaves 4 tracks with a hole around the crossing. Before you run the cell:
`max_gap=60` can jump the hole; does the rising glide carry on upward?

In [ ]:
bridged = asp.link_peaks(peaks, max_gap=60, min_duration=0.1)
print(f"max_gap=60: {count(bridged)} tracks")
for tid, g in bridged.groupby("track_id"):
    print(f"track {tid}: {g.freq.iloc[0]:.1f} to {g.freq.iloc[-1]:.1f} Hz, "
          f"{g.time.min():.2f}–{g.time.max():.2f} s, "
          f"range {g.freq.min():.1f}–{g.freq.max():.1f} Hz")
parts = [g for _, g in bridged.groupby("track_id")]
asp.nb.show(asp.viz.plot_events(*parts, labels=[f"track {i}" for i in range(len(parts))]))

It turns back. With `max_gap=60` there are 3 tracks: track 0 starts at
497.2 Hz and ends at 501.0 Hz, while track 1 starts at 998.6 Hz and ends at
999.3 Hz. A short middle track spans 689.7–707.8 Hz from 0.90 to 1.10 s. The
default, `prediction="hold"`, expects a track at its last measured frequency;
it carries no information that the glide was rising. These paths follow that
prediction, not the identities of the two sines. `prediction="lp"` instead
continues each track's recent motion through the missed frames.

In [ ]:
carried = asp.link_peaks(peaks, max_gap=60, min_duration=0.1, prediction="lp")
print(f'prediction="lp": {count(carried)} tracks')
for tid, g in carried.groupby("track_id"):
    print(f"track {tid}: {g.freq.iloc[0]:.1f} to {g.freq.iloc[-1]:.1f} Hz, "
          f"{g.time.min():.2f}–{g.time.max():.2f} s")
parts = [g for _, g in carried.groupby("track_id")]
asp.nb.show(asp.viz.plot_events(*parts, labels=[f"track {i}" for i in range(len(parts))]))

Track 0 now runs from 497.2 Hz to 999.3 Hz and track 1 from 998.6 Hz to
501.0 Hz: each glide carries on through the crossing. That is right only if
the glides do cross. Two glides can also meet and turn back, with the same
ends and the same samples up to the meeting at 1 s. Before you run the cell:
which prediction follows them?

In [ ]:
m = np.minimum(t, 2 - t)
back = 0.3 * np.sin(2 * np.pi * np.cumsum(500 * 2 ** (m / 2)) / sr)
back += 0.3 * np.sin(2 * np.pi * np.cumsum(1000 * 2 ** (-m / 2)) / sr)
spec_back = asp.spectrogram(back, sr, window=BH, hop=256, zero_pad=4)
peaks_back = asp.spectrogram_peaks(spec_back, min_db=-80, prominence_db=20)
rows = []
for name, table in [("crossing", peaks), ("turn-back", peaks_back)]:
    for prediction in ["hold", "lp"]:
        tr = asp.link_peaks(table, max_gap=60, min_duration=0.1, prediction=prediction)
        rows.append({"glides": name, "prediction": prediction,
                     "track 0 ends (Hz)": round(tr[tr.track_id == 0].freq.iloc[-1], 1)})
pd.DataFrame(rows)

Each prediction gives one answer for both pairs. `hold` ends track 0 at
501.0 and 500.9 Hz: wrong for the crossing, right for the turn-back. `lp` ends
it at 999.3 and 999.2 Hz: right for the crossing, wrong for the turn-back.
Compare the two peak tables to see why.

In [ ]:
two = peaks.groupby("frame").freq.transform("size").to_numpy() == 2
apart = np.abs(peaks.freq.to_numpy() - peaks_back.freq.to_numpy())
print(f"{len(peaks)} and {len(peaks_back)} peaks, same frames: "
      f"{peaks.frame.equals(peaks_back.frame)}")
print(f"largest difference: {apart[two].max():.2f} Hz in two-peak frames, "
      f"{apart[~two].max():.1f} Hz in one-peak frames")

Both tables have 617 peaks in the same frames. Where two peaks stand apart
they differ by at most 0.11 Hz. The one-peak frames differ by up to 64.8 Hz,
but one peak is one frequency: it cannot show which partial went where. So a
prediction is an assumption, not a measurement. `lp` suits partials known to
keep moving through a gap, `hold` partials that may turn.
[`benchmarks/accuracy/`](../benchmarks/accuracy/README.md#meetings-and-holes)
counts both cases on matched pairs and on holes in moving partials, with what
a large `max_gap` costs in noise.

## Short tracks in noise

A noise peak can survive several overlapping frames. `min_duration` removes
short paths after linking; it cannot tell noise from a short musical partial.
The same peaks are used for every duration below.

In [ ]:
x = np.random.default_rng(0).normal(0, 0.1, sr)
spec = asp.spectrogram(x, sr, hop=256, zero_pad=4)
peaks = asp.spectrogram_peaks(spec, min_db=-80, prominence_db=20)
print(f"{len(peaks)} peaks")
rows = []
for d in [0, 0.02, 0.05, 0.1, 0.2]:
    tr = asp.link_peaks(peaks, min_duration=d)
    rows.append({"min_duration_s": d, "tracks": count(tr)})
pd.DataFrame(rows)

8,972 peaks become 2,762, 617, 164, 24 and 0 tracks as the minimum span
rises through 0, 0.02, 0.05, 0.1 and 0.2 s. The `track_partials` default of
0.1 s comes from this trade-off: a real partial shorter than 0.1 s goes the
same way. The defaults were chosen on the development suite in
[`benchmarks/accuracy/`](../benchmarks/accuracy/README.md), not on the flute
below.

## A low rumble

Band-limit noise to 30–90 Hz, then scale its peak to 0.5. Low, noisy material
can be rejected both by the prominence floor and by the narrow cents gate.
Before you run the cell: what will `track_partials` return with its defaults?

In [ ]:
rng = np.random.default_rng(3)
x = rng.normal(0, 1, 3 * sr)
freq = np.fft.rfftfreq(len(x), 1 / sr)
spectrum = np.fft.rfft(x)
spectrum[(freq < 30) | (freq > 90)] = 0
x = np.fft.irfft(spectrum, n=len(x))
x *= 0.5 / np.max(np.abs(x))
print(f"rumble RMS {rms_db(x):.1f} dB")
empty = asp.track_partials(x, sr)
render = asp.synthesize(asp.tracks_to_events(empty), sr)
print(f"defaults: {count(empty)} tracks, {len(empty)} rows, {len(render)} samples")

An empty table, and a warning: 560 peaks, no track. The warning names the
settings to try; `link_peaks` alone does not warn. The rumble has RMS −17.0 dB,
but the resynthesis has 0 samples. Guard a player with `if audio.size:` before
calling `asp.nb.play`. Compare removing the prominence floor, widening the
gate, both together, and `min_gate_hz`, a floor under the gate in Hz.

In [ ]:
rows = []
for label, tr in [
    ("defaults", empty),
    ("prominence_db=None", asp.track_partials(x, sr, prominence_db=None)),
    ("max_cents=400", asp.track_partials(x, sr, max_cents=400)),
    ("both", asp.track_partials(x, sr, prominence_db=None, max_cents=400)),
    ("min_gate_hz=12", asp.track_partials(x, sr, min_gate_hz=12)),
]:
    audio = asp.synthesize(asp.tracks_to_events(tr), sr)
    rows.append({"setting": label, "tracks": count(tr), "RMS_dB": round(rms_db(audio), 1)})
print(f"50 cents at 60 Hz: {60 * (2 ** (50 / 1200) - 1):.2f} Hz; grid {step:.2f} Hz")
pd.DataFrame(rows)

Removing the prominence floor gives 11 tracks at −62.8 dB RMS; a 400-cent
gate gives 6 at −23.0 dB; both give 52 at −20.5 dB; a 12 Hz floor under the
gate gives 6 at −21.7 dB. The default 50-cent gate at 60 Hz is only 1.76 Hz,
against a grid step of 5.38 Hz: the gate that suits a flute is narrower here
than one step of the grid. Whichever setting lets the rumble through, its
tracks follow noise paths, not partials. See
[`benchmarks/fidelity/README.md`](../benchmarks/fidelity/README.md) for the
same silent render on a real recording and what a wide gate costs elsewhere.
This is no recommendation to make `max_cents=400` the default: lower measured
error has not been shown to sound better.

## The flute next to PM2

Load the Saariaho flute excerpt and its harmonic PM2 analysis, described in
[`data/interop/README.md`](../data/interop/README.md). PM2 is a comparison,
not the truth: both analyses are estimates, made with different settings.

In [ ]:
x, sr = asp.load_audio("../samples/saariaho_flute.wav", mono=True)
pm2 = asp.read_sdif("../data/interop/saariaho_flute_pm2.sdif")
tracks = asp.track_partials(x, sr)
for label, tr in [("PM2", pm2), ("track_partials", tracks)]:
    times = np.sort(tr.time.unique())
    print(f"{label}: {count(tr)} tracks, {len(tr)} rows, "
          f"{tr.freq.min():.1f}–{tr.freq.max():.1f} Hz, "
          f"frames {1000 * np.median(np.diff(times)):.2f} ms apart")
spec = asp.spectrogram(x, sr, hop=256, zero_pad=4)
asp.nb.show(asp.viz.plot_spectrogram(spec, overlay=[pm2, tracks],
                                   overlay_labels=["PM2", "track_partials"],
                                   overlay_mode=["markers", "lines"], fmax=6000))

PM2 has 398 tracks and 17,310 rows over 156.8–13,609.2 Hz, with frames
7.76 ms apart. `track_partials` has 156 tracks and 9,422 rows over
200.5–8,478.8 Hz, with frames 5.80 ms apart. The plot draws PM2 as
points and the tracker as lines, and crops at 6000 Hz; the printed range
covers the whole table. To measure agreement, count a point
when the other analysis has any point within 50 cents and 20 ms. This is a
local proximity test, without requiring the same track identity.

In [ ]:
def agrees(a, b, cents=50, seconds=0.020):
    b = b.sort_values("time")
    bt, bf = b.time.to_numpy(), b.freq.to_numpy()
    result = []
    for row in a.itertuples():
        lo = np.searchsorted(bt, row.time - seconds, side="left")
        hi = np.searchsorted(bt, row.time + seconds, side="right")
        distance = np.abs(1200 * np.log2(bf[lo:hi] / row.freq))
        result.append(np.any(distance <= cents))
    return np.array(result)

Time bounds select the nearby frames; the cents test then checks their
frequencies. Count the points and their squared amplitudes separately, since
a quiet point and a loud one otherwise count equally.

In [ ]:
pm2_near = agrees(pm2, tracks)
tracker_near = agrees(tracks, pm2)
for label, tr, near in [("PM2 near tracker", pm2, pm2_near),
                        ("tracker near PM2", tracks, tracker_near)]:
    energy = tr.amp.to_numpy() ** 2
    print(f"{label}: {100 * near.mean():.1f}% points, "
          f"{100 * energy[near].sum() / energy.sum():.1f}% energy")
for lo, hi in [(0, 1000), (1000, 3000), (3000, 6000), (6000, np.inf)]:
    band = (pm2.freq >= lo) & (pm2.freq < hi)
    print(f"PM2 {lo:g}–{hi:g} Hz: {100 * pm2_near[band].mean():.1f}% points agree")

41.2% of PM2 points agree with the tracker, carrying 97.1% of PM2's
squared-amplitude energy. In the other direction, 97.9% of tracker points
agree, carrying 99.9% of its energy. The tracker misses mostly quiet PM2
points, and almost all it keeps PM2 also has. PM2 point agreement falls by
band: 70.4% at 0–1 kHz, 42.5% at 1–3 kHz, 32.6% at 3–6 kHz and 19.8%
above 6 kHz. These are agreement figures; neither analysis supplies a known
answer. Track spans show a different aspect of the comparison.

In [ ]:
for label, tr in [("tracker", tracks), ("PM2", pm2)]:
    spans = tr.groupby("track_id").time.agg(["min", "max"])
    print(f"{label} longest track: {(spans['max'] - spans['min']).max():.2f} s")
spans = tracks.groupby("track_id").time.agg(["min", "max"])
duration = spans["max"] - spans["min"]
low = tracks.groupby("track_id").freq.median().between(580, 600)
for tid in [duration.idxmax(), duration[low].idxmax()]:
    g = tracks[tracks.track_id == tid]
    print(f"tracker {tid}: first {g.freq.iloc[0]:.0f} Hz, last {g.freq.iloc[-1]:.0f} Hz, "
          f"from {g.time.min():.2f} s, span {duration[tid]:.2f} s")

The longest track spans 3.09 s for the tracker, 7.36 s for PM2. The
tracker's longest starts at 1178 Hz and ends at 1182 Hz, from 4.29 s;
the longest whose median lies in 580–600 Hz starts at 582 Hz and ends at
596 Hz, from 4.17 s. These endpoints lie close to the notes that stay put in
`tracks_chords_mei`. A minimum duration sets a floor under the tracker spans,
so their distribution cannot be compared fairly without accounting for that
filter. The two longest spans differ in kind as well: check how far a track
moves in one step.

In [ ]:
def longest_run(tr, cents=50):
    best = 0.0
    for _, g in tr.groupby("track_id"):
        g = g.sort_values("time")
        jump = np.abs(1200 * np.diff(np.log2(g.freq.to_numpy()))) >= cents
        spans = g.groupby(np.r_[0, np.cumsum(jump)]).time.agg(["min", "max"])
        best = max(best, (spans["max"] - spans["min"]).max())
    return best

for label, tr in [("tracker", tracks), ("PM2", pm2)]:
    s = np.concatenate([np.abs(1200 * np.diff(np.log2(g.sort_values("time").freq.to_numpy())))
                        for _, g in tr.groupby("track_id")])
    print(f"{label}: {(s >= 50).sum()} of {len(s)} steps reach 50 cents "
          f"({100 * (s >= 50).mean():.1f}%), largest {s.max():.0f} cents, "
          f"longest run below that {longest_run(tr):.2f} s")
first = pm2[pm2.track_id == 0]
print(f"PM2 track 0: {first.freq.min():.0f}–{first.freq.max():.0f} Hz")

A PM2 track here is not one steady partial. Track 0 ranges over 157–928 Hz,
and 5,434 of PM2's 16,912 steps (32.1%) reach 50 cents, the largest 2600
cents; none of the tracker's 9,266 steps do, because its gate excludes them.
Cut at those steps, PM2's longest run is 1.14 s against the tracker's 3.09 s.
Neither figure ranks the two analyses: they define a track differently, so
read the counts of 398 and 156 tracks with the same care.

## Settings on the flute

Change one setting at a time. Lowering or removing the prominence floor keeps
more peaks; changing the window changes which partials can be separated.
Before you run the cell: does `max_gap=2` give more tracks or fewer than the
default?

In [ ]:
bh = asp.track_partials(x, sr, window=BH)
prom10 = asp.track_partials(x, sr, prominence_db=10)
noprom = asp.track_partials(x, sr, prominence_db=None)
gap2 = asp.track_partials(x, sr, max_gap=2)
rows = [{"setting": label, "tracks": count(tr), "rows": len(tr)}
        for label, tr in [("defaults", tracks), ("window=BH", bh),
                          ("prominence_db=10", prom10), ("prominence_db=None", noprom),
                          ("max_gap=2", gap2)]]
print(f"prominence_db=None: lowest {noprom.freq.min():.1f} Hz")
pd.DataFrame(rows)

More: `max_gap=2` gives 226 tracks and 12,972 rows, against 156 and 9,422
at the defaults. Blackman-Harris gives 128 and 7,209; a 10 dB prominence
floor gives 281 and 14,362; removing it gives 773 and 31,303, reaching down
to 7.0 Hz. More tracks need not mean more distinct partials. Reuse one peaks
table to separate linking from the duration filter.

In [ ]:
peaks = asp.spectrogram_peaks(spec, min_db=-80, prominence_db=20)
print(f"same peaks table: {len(peaks)} rows")
for duration in [0, 0.1]:
    n0 = count(asp.link_peaks(peaks, max_gap=0, min_duration=duration))
    n2 = count(asp.link_peaks(peaks, max_gap=2, min_duration=duration))
    print(f"min_duration={duration:g} s: max_gap=0 {n0} tracks; max_gap=2 {n2} tracks")

On the same 20,677 peaks, allowing gaps joins pieces: before duration
filtering, 4,398 tracks become 2,669. After the 0.1 s filter, 156 become
226 because joined pieces now survive it. The apparent reversal comes from
the filter, not from the linker making more pieces.

## Listen

Resynthesise both analyses through `tracks_to_events` and `synthesize` so they
share the same renderer. Each step between two points of a track becomes a
glide, so PM2's jumps play as fast glides. Each player uses `normalize=True`; the printed
levels are before that scaling. Listen for the breath and the attacks: what
does each resynthesis keep of them?

In [ ]:
tracker_audio = asp.synthesize(asp.tracks_to_events(tracks), sr)
pm2_audio = asp.synthesize(asp.tracks_to_events(pm2), sr)
for label, audio in [("recording", x), ("track_partials", tracker_audio), ("PM2", pm2_audio)]:
    print(f"{label}: peak {np.max(np.abs(audio)):.3f}, RMS {rms_db(audio):.2f} dB")
    asp.nb.play(audio, sr, normalize=True)

Recording, tracker and PM2 have peaks 0.062, 0.059 and 0.061, and RMS
levels −36.18, −36.56 and −36.21 dB. Similar levels and agreement on strong
points do not establish which resynthesis sounds closer: no listening test
backs a result here. Use the players to compare what each keeps of the breath
and the attacks.